# Overview
This jupyter notebook constructs a fictional EMG dataset in BIDS format using the "muniverse" ( https://github.com/dfarinagroup/muniverse/ ) framework. 

The dataset contains both surface and invasive EMG. At the time of making this notebook, invasive EMG is not yet officially supported by BIDS. Here a custom approach is used, where the information about invasive EMG is encoded in additional columns in the electrodes.tsv files in a way that is consistent with the BIDS standard. 

The experimental setup is intentionally more complex than most regular datasets for illustrative purposes. There are 3 Grids of HDsEMG. There are 2 Grids of HDiEMG (thin filaments). Additionally there are 2 Fine-Wire Electrodes and 2 Needle Electrodes, as well as 2 reference electrodes and 1 ground electrode. Grids have different numbers of rows and columns. The dataset contains 6 subjects who each performed 3 tasks (one "restNoise" task (i.e. baselineNoise task), and two contraction tasks called "trapezoidalContraction30PercentMVC" and "trapezoidalContraction50PercentMVC"). 

<img src="fictionalDatasetExampleInputMetadata/experimentalSetup.jpg" alt="Drawing" style="width: 700px;"/> <img src="fictionalDatasetExampleInputMetadata/schematics.jpg" alt="Drawing" style="width: 500px;"/>


This is what the final folder structure of this dataset looks like. The actual data is contained in *.edf* files, one for each task. Since *.edf* files are just a matrix of numbers, accompanying metadata files are needed. This is what all other files are. 

There are some metadata files that describe the dataset as a whole, like *dataset_description.json*, *participants.tsv* and *README.md*. 

There are other metadata files which describe individual recordings. The most important ones are *_channels.tsv* (which tells you which row of the matrix contains what kind of data and corresponds to which electrodes) and *_emg.json* (which tells you everything from how electrodes were placed, to intructions given to the participant, to details of the measurement hardware). For very simple experimental setups these two suffice. However for more complex setups an *_electrodes.tsv* is needed to specify things like electrode position, type and material. If *_electrodes.tsv* exists we also need *_coordsystem.json* files to create coordinate systems which give meaning to the electrode position specified in *_electrodes.tsv*. If events are tracked we need *_events.tsv* files. 

The files *electrodes.json*, *events.json*, *participants.json* and *channels.json* only exist to specify any additional non-default columns we may have introduced in the respective *.tsv* files. 

Files like *_emg.json* must exist for each *.edf* file, while other recording specific metadata can be inherited if its the same for every recording, such as *_electrodes.tsv* on a subject level and *_electrodes.json* on a dataset level in our case. More on inheritance later. 
```
FictionalDatasetExample/
├── dataset_description.json
├── electrodes.json
├── events.json
├── participants.json
├── participants.tsv
├── README.md
├── sub-01/
│   └── emg/
│       ├── sub-01_electrodes.tsv
│       ├── sub-01_space-forearmCoordSys_coordsystem.json
│       ├── sub-01_space-grid1CoordSys_coordsystem.json
│       ├── sub-01_space-grid2CoordSys_coordsystem.json
│       ├── sub-01_space-grid3CoordSys_coordsystem.json
│       ├── sub-01_space-intraGrid1CoordSys_coordsystem.json
│       ├── sub-01_space-intraGrid2CoordSys_coordsystem.json
│       ├── sub-01_task-restNoise_run-01_channels.json
│       ├── sub-01_task-restNoise_run-01_channels.tsv
│       ├── sub-01_task-restNoise_run-01_emg.edf
│       ├── sub-01_task-restNoise_run-01_emg.json
│       ├── sub-01_task-trapezoidalContraction30PercentMVC_run-01_channels.json
│       ├── sub-01_task-trapezoidalContraction30PercentMVC_run-01_channels.tsv
│       ├── sub-01_task-trapezoidalContraction30PercentMVC_run-01_emg.edf
│       ├── sub-01_task-trapezoidalContraction30PercentMVC_run-01_emg.json
│       ├── sub-01_task-trapezoidalContraction30PercentMVC_run-01_events.tsv
│       ├── sub-01_task-trapezoidalContraction50PercentMVC_run-01_channels.json
│       ├── sub-01_task-trapezoidalContraction50PercentMVC_run-01_channels.tsv
│       ├── sub-01_task-trapezoidalContraction50PercentMVC_run-01_emg.edf
│       ├── sub-01_task-trapezoidalContraction50PercentMVC_run-01_emg.json
│       └── sub-01_task-trapezoidalContraction50PercentMVC_run-01_events.tsv
├── sub-02/
│   └── ...
```
This notebook will go through the construction of all files, starting with dataset wide metadata, then recording specific metadata, and finally the *.edf* files. Finally the bids-validator is used to verify if the dataset adheres to BIDS rules. 

In [ ]:
import numpy as np
import pandas as pd
import muniverse.utils.bids_routines
import copy

# Metadata 
## Global (dataset wide) metadata
### Subjects data & sidecar
We define the data of our subjects. This will end up in the file "participants.tsv". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/data-summary-files.html#participants-file 

In [ ]:
subjects_data = {
    "participant_id": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06"], # Unique subject identifier. Every name must start with "sub-"
    "age": [42, 43, 44, 45, 46, 47], # optional 
    "sex": ["M", "F", "M", "F", "M", "F"], # optional 
    "handedness": ["R", "R", "R", "R", "L", "R"], # optional 
    "weight": [70, 68, 66, 64, 62, 60], # optional 
    "height": [1.7, 1.72, 1.74, 1.76, 1.78, 1.8], # optional 
    "group": ["T", "T", "T", "P", "P", "P"], # optional 
}

We define the subjects sidecar file. This information will end up in "participants.json". 
We only need to do this for extra columns, since the following defaults are already added by muniverse: "participant_id", "age", "sex", "handedness", "weight", "height". 

In [ ]:
subjects_sidecar = {
    "group": {
        "Description": "Group the subject belongs to.",
        "Levels": {
            "T": "Treatment", 
            "P": "Placebo"
        }
    }
}

### Dataset sidecar 
We define the dataset sidecar, which contains some general information about the dataset. This will end up in "dataset_description.json". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#dataset_descriptionjson 

In [ ]:
dataset_sidecar = {
  "Name": "DatasetName",
  "License": "CC BY 4.0", # License this dataset will be available under.
  "Authors": [
    "alice",
    "bob"
  ], # List of individuals who contributed to the creation/curation of the dataset., has to be a list even if it has only one entry 
  "ReferencesAndLinks": [
    "citation of related publication as text",
    "related publication as DOI"
  ], # Citation of the related publication, as text and as DOI. 
  "EthicsApprovals": [
    "number of ethics approval."
  ], # List of ethics committee approvals of the research protocols and/or protocol identifiers. Has to be a list even if it has only one entry 
  "GeneratedBy": [{"Name":"munitquest tutorials"}], # must be array of objects 
}

### Readme File 
We define a readme for the dataset. This becomes "readme.md". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#readme 

Ideally the readme gives a thorough overview of your dataset. This ranges from general information on how to access the data, a description of independent, dependent and control variables, experimental setup and task descriptions, quality assessment, missing data points, to information that did not fit into any of the other metadata files. 

A readme Template can be found here: https://bids.neuroimaging.io/getting_started/templates/index.html?h=readme#readmemd 


In [ ]:
readme = """
Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.

Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
"""

### Tasks
We define the names of our tasks the participants performed. 

In [ ]:
tasklist = ["restNoise", "trapezoidalContraction30PercentMVC", "trapezoidalContraction50PercentMVC"]

## Local (recording specific) metadata
### EMG sidecar 
We define the emg sidecar. This will end up in files ending with "..._emg.json". There will be one such file per recording. 
In our case they are identical for each recording, except for task related information such as *Taskname* and *TaskDescription* and *Instructions*. 
More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#sidecar-json-_emgjson 

In [ ]:
emg_sidecar_task1 = {
    "EMGPlacementScheme": "Measured", # Must be the keyword "Measured" if you have grids and specify electrode locations in coordinate systems. 
    "EMGPlacementSchemeDescription": "(i) Surface EMG: lorem ipsum. (ii) Invasive thin film EMG: lorem ipsum. (iii) Invasive Fine Wire EMG: lorem ipsum. (iv) Invasive Needle EMG: lorem ipsum.", # Describe how electrodes are placed. Include anatomical landmarks used to position. Include the measurement method for placement. Include placement of reference electrode(s). Include placement of ground electrode. Include if a dry linear array for fiber alignment was used or not. Include if innervation zone was measured and how electrodes are positioned relative to it. For different types of electrodes (surface grid, invasive grid, fine wire, etc) use i), ii), iii), ... to separate placement description (similar to our provided example). 
    "EMGReference": "ChannelSpecific",
    "EMGGround": "G1", # The name of the ground electrode (as specified in electrodes metadata). 
    "SamplingFrequency": 2048, # The main sampling frequency (in Hz) of your data. If your dataset contains more than one sampling frequency contact us. 
    "PowerLineFrequency": 50, # Frequency (in Hz) of the power grid where the data was recorded. 
    "RecordingType": "continuous", 
    "HardwareFilters": {
        "Anti-aliasing filter": {
            "half-amplitude cutoff (Hz)": 500,
            "Roll-off": "6dB/Octave"
        },
        "some other filter": {
            "filterparameter 1": 500,
            "filterparameter 2": "12dB/Octave",
            "filterparameter 3": 80
        }
    }, # A json object containing filter parameters. Use "n/a" if no filter was used.
    "SoftwareFilters": "n/a", # A json object containing filter parameters. Use "n/a" if no filter was used.
    "EMGChannelCount":45,
    "TaskName": "restNoise",
    "TaskDescription": "Relaxed muscle for 5 seconds.",
    "Instructions": "Relax your muscle completely.", 
    "Preamplification": 1, # Amplification built into an EMG bipolar sensor, electrode grid, or other device.
    "Gain": 1, # Signal gain from an in-line amplifier, applied between the EMG sensor/device and the data acquisition computer. 
    "Manufacturer": "some amplifier manufacturer", # Manufacturer of the amplifier used to collect the data. 
    "ManufacturersModelName": "some amplifier model name" # Model name of the amplifier. 
}

# Since almost everything is the same for the other tasks we create a copy and only change task related fields. 
emg_sidecar_task2 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task2["TaskName"] = "trapezoidalContraction30PercentMVC"
emg_sidecar_task2["TaskDescription"] = "A trapezoidal contraction at 30 percent MVC, consisting of linear ramps up and down performed at 30 percent per second and a plateau maintained for 1 s."
emg_sidecar_task2["Instructions"] = "Follow path provided via visual feedback."

emg_sidecar_task3 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task3["TaskName"] = "trapezoidalContraction50PercentMVC"
emg_sidecar_task3["TaskDescription"] = "A trapezoidal contraction at 50 percent MVC, consisting of linear ramps up and down performed at 50 percent per second and a plateau maintained for 1 s."
emg_sidecar_task3["Instructions"] = "Follow path provided via visual feedback."

# To retrieve these cleanly later from inside a for-loop we pack them into a dict with tasknames as keys 
emg_sidecar_taskDict = {
    "restNoise": emg_sidecar_task1, 
    "trapezoidalContraction30PercentMVC": emg_sidecar_task2, 
    "trapezoidalContraction50PercentMVC": emg_sidecar_task3, 
}

### Coordinate system sidecar 
If we want to specify electrodes positions on the body, BIDS requires us to define the coordinate systems our electrodes will be positioned inside. For a grid of electrodes, the recommended approach is to define electrode positions in a local grid-specific coordinate system, and then locate the grid specific coordinate system inside an anatomical parent coordinate system. We do this for every grid. Both child and parent coordinate systems end up in files ending with "_coordsystem.json". 

Our parent coordinate system is defined through anatomical landmarks on the forearm, with percent between landmarks as the coordinate unit. The grid specific systems are then located by designating an anchor electrode and its location in the parent coordinate system. 

More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#coordinate-system-json-_coordsystemjson 

The actual information of where each electrode is happens in the next step, inside the *_electrodes.tsv* files. 

In [ ]:
coord_sidecar = {
    "forearmCoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "percent",
        "EMGCoordinateSystemDescription": "x: Radial Styloid Process (RSP) -> Ulnar Styloid Process (USP); y: Right-hand rule (limits: Olecranon Process -> Cubital Fossa); z: midpoint RSP-USP -> Lateral Humerus Epicondyle (LHE)"
    }, 

    "grid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": " The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E001",
        "AnchorCoordinates": [30, 50, 80]
    }, 

    "grid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E009",
        "AnchorCoordinates": [30, 50, 60]
    }, 

    "grid3CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E017",
        "AnchorCoordinates": [30, 50, 40]
    }, 

    "intraGrid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE001",
        "AnchorCoordinates": [30, 50, 70]
    }, 

    "intraGrid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE009",
        "AnchorCoordinates": [30, 50, 50]
    }
}

### Electrode data & sidecar
This is where we locate our electrodes inside our coordinate systems and specify additional electrode information, such as type, material, diameter or surface area, manufacturer, etc. 

We load electrode metadata from a file. This becomes files called "..._electrodes.tsv". Instead of loading from a file you could also write code that builds the desired dataframe. More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#electrodes-description-_electrodestsv 

Some notes on this: 
- Electrode names must be unique. 
- "x", "y" and "z" specify position inside "coordinate_system" (whose names must match the defined coordinate systems), "z" can usually be left empty. 
- If you have fine-wires, please specify the diameter of the fine wire tip in "electrode_diameter", as well as unisolated length of the fine wire tip in "electrode_tip_length". 
- "interelectrode_distance" in a grid means distance between neighboring electrodes, while in a fine wire it means distance between the wire tips. 
- If you have concentric needles, please specify length and diameter in the columns "cannula_length" and "cannula_diameter". 

In [ ]:
el_metadata = pd.read_csv("fictionalDatasetExampleInputMetadata//electrodes.tsv", sep="\t") 
el_metadata.head()

We define the electrodes sidecar. This becomes "..._electrodes.json" files. Similar to the subjects sidecar we only need to define the non-default columns. 

In [ ]:
electrodes_sidecar = {
    "interelectrode_distance": "Distance between electrodes. In a grid this means distance between neighboring electrodes. In a fine wire it means distance between the wire tips.", 
    "electrode_surface_area": "Surface area of the electrode", 
    "electrode_diameter": "Diameter of the electrode", 
    "electrode_tip_length": "Unisolated length of the fine wire tip", 
    "cannula_diameter": "Diameter of Cannula", 
    "cannula_length": "Length of Cannula", 
    "manufacturer": "Name of electrode manufacturer", 
    "manufacturers_model_name": "Model name of Electrode", 
}

### Channel data & sidecar
This is where we specify channel information, such as name, type and unit. We also specify which electrodes are associated with which channel, as well as some information about channel quality and applied filters. 

We load channel data from a file. This becomes "..._channels.tsv". More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#channels-description-_channelstsv 

Some notes on this: 
- channel names must be unique. The electrode names in the "signal_electrode" and "reference_electrode" columns must exist in electrodes metadata. 
- The Type of channel must be "EMG" or "MISC" or "TRIG". 
- "status" can be used to tag channels which should be ignored in data analysis. Must be "good", "bad" or left empty. 
- "low_cutoff" and "high_cutoff" refers to Low/High-pass filter frequency (in Hz). 

In [ ]:
ch_metadata = pd.read_csv("fictionalDatasetExampleInputMetadata//channels.tsv", sep="\t") 
ch_metadata.head()

We define channels sidecar. This becomes "..._channels.json" files. Again we only need to define non-default columns. 

In [ ]:
channels_sidecar = {
    "some_additional_column": "some Description of this column" # we define an optional column for illustrative purposes
}

### Events data & sidecar
We load event data from a file. This becomes "..._events.tsv". 

Here we describe timing and other properties of events recorded during data acquisition. Generally events can be things like: presented stimuli presented, participant responses, or incident markers. 
For more information see: https://bids-specification.readthedocs.io/en/stable/modality-agnostic-files/events.html 

Here we use events to mark the start and end of contraction, as well as the beginning of a contraction-ramp-up, contraction-ramp-down or contraction-hold phase. Besides just timing we also include information such as how long the phase lasts and what level of contraction (in %MVC) there is, as well as how fast this level changes (for ramp-up and down phases). 

In [ ]:
# the baseline noise recording does not get an events.tsv file, because there are no events we could write into it. 
events_metadata1 = pd.read_csv("fictionalDatasetExampleInputMetadata//events30PercentMVC.tsv", sep="\t")
events_metadata2 = pd.read_csv("fictionalDatasetExampleInputMetadata//events50PercentMVC.tsv", sep="\t")
# To retrieve these cleanly later from inside a for-loop we pack them into a dict with tasknames as keys 
events_metadata_taskDict = {
    "trapezoidalContraction30PercentMVC": events_metadata1, 
    "trapezoidalContraction50PercentMVC": events_metadata2,    
}
events_metadata1.head()

We define events sidecar. This becomes "events.json". Again we need to define non-default columns. 

In [ ]:
events_sidecar = {
    "sample": {
        "Description": "Sample index of the event onset (zero-indexing)",
        "Unit": "samples"
    },
    "mvc_rate": {
        "Description": "Rate at which the torque changes in percent MVC per second",
        "Unit": "% MVC / s"
    },
    "mvc_level": {
        "Description": "MVC (maximum voluntary contraction) level at the onset of the event",
        "Unit": "% MVC"
    },
    "event_type": {
        "Description": "Event label.",
        "Levels": {
            "muscle_on": "The muscle is activated.",
            "muscle_off": "The muscle is deactivated.",
            "linear_isometric_ramp": "The isometric torque changes linearly over time with a fixed rate.",
            "steady_isometric": "Steady isometric contraction at a fixed MVC level."
        }
    },
    "description": {
        "Description": "Free text event description."
    }
}

# Building the dataset
Now we create an instance of a muniverse bids_dataset. 
First we set global metadata. 

In [ ]:
FictionalDatasetExample = muniverse.utils.bids_routines.bids_dataset(datasetname="FictionalDatasetExample")
FictionalDatasetExample.set_metadata(field_name='subjects_data', source=subjects_data)
FictionalDatasetExample.set_metadata(field_name='subjects_sidecar', source=subjects_sidecar)
FictionalDatasetExample.set_metadata(field_name='dataset_sidecar', source=dataset_sidecar)
FictionalDatasetExample.readme = readme
FictionalDatasetExample.write()

Then we loop over all participants and tasks, create an instance of bids_emg_recording for each, add all local metadata and the raw data to it. In this case we randomly generate the raw data, but you could easily replace that with a function like *get_raw_data(participant_id, task)* to insert your own data. 

Some metadata is the same for all tasks within a subject or even for all subjects. Instead of writing a redundant file for each subject and each task we can use inheritance to save some discspace. This is done with the arguments *inherited_metadata* and *inherited_level* that are passed to the *bids_emg_recording* class. 

For more information about inheritance see: https://bids-specification.readthedocs.io/en/stable/common-principles.html#the-inheritance-principle (inheritance can also be used if for example one measurement of one subjects differs from all the rest) 

In [ ]:
rng = np.random.default_rng(seed=12345)

for participant_id in subjects_data["participant_id"]:
    participant_id = participant_id[4:]
    print(f"Bidsifying data of sub-{participant_id}")

    for task in tasklist:
        # create a random array of the correct size to be our raw data. 
        n_channels = len(ch_metadata.loc[:,"name"])
        recordingLength = 5 # seconds 
        samplingFrequency = 2048 # samples per second 
        n_samples = np.ceil(recordingLength * samplingFrequency)
        data = rng.uniform(low=0, high=1, size=(n_channels, n_samples))

        # Make a recording and add data and metadata
        emg_recording = muniverse.utils.bids_routines.bids_emg_recording(
            parent_dataset = FictionalDatasetExample, 
            subject_label=participant_id, 
            task_label=task, 
            datatype='emg',
            inherited_metadata=["coordsystem.json", "electrodes.tsv", "electrodes.json", "events.json"], # Here is where we define which files to inherit
            inherited_level=["subject", "subject", "dataset", "dataset"], # and here which level they are inherited to 
        )
        emg_recording.set_metadata(field_name='emg_sidecar', source=emg_sidecar_taskDict[task])
        emg_recording.set_data(field_name='emg_data', mydata=data,fsamp=samplingFrequency)
        emg_recording.set_metadata(field_name='coord_sidecar', source=coord_sidecar, overwrite=True)
        emg_recording.set_metadata(field_name='electrodes_sidecar', source=electrodes_sidecar)
        emg_recording.set_metadata(field_name='electrodes', source=el_metadata) 
        emg_recording.set_metadata(field_name='channels_sidecar', source=channels_sidecar)
        emg_recording.set_metadata(field_name='channels', source=ch_metadata)

        emg_recording.set_metadata(field_name="events_sidecar", source=events_sidecar)
        if task in events_metadata_taskDict.keys(): 
            emg_recording.set_metadata(field_name="events", source=events_metadata_taskDict[task])
        emg_recording.write()


# Validation
Finally we use the BIDS-validator to check the integrity of our dataset. 
For more information see here: https://github.com/bids-standard/bids-validator 

The muniverse wrapper for the bids-validator allows us to ignore certain codes or fields. This can be helpful for decluttering the output of the validator. 

In [ ]:
err, warn, _ = FictionalDatasetExample.validate(
    print_errors=True,
    print_warnings=True,
    ignored_codes=[
        "EVENTS_TSV_MISSING", # we specify events.tsv files, but not for restNoise. In order to not get one warning per restNoise recording in the dataset, we ignore the code. 
    ],
    ignored_fields=[ # We ignore some fields that would produce warnings
        "SourceDatasets", # there isn't one
        "DeviceSerialNumber", # no hardware was used to record this dataset
        "SoftwareVersions", # this also pertains to measurement hardware 
        "InstitutionName", # we also don't want to specify an affiliated institution 
        "InstitutionAddress",
        "InstitutionalDepartmentName",

        "HEDVersion", # we don't use HED. 
        "StimulusPresentation", # we also ignore this, to not clutter this jupyter notebook. 

        # We have different ElectrodeManufacturer and ElectrodeManufacturersModelName per electrode, so we specify it on a per electrode basis in the electrodes.tsv file, rather than inside of emg.json (as mandated by the BIDS documentation for this case). However at the time of writing this script, the Validator is not smart enough to understand this. So we ignore the raised warning. 
        "ElectrodeManufacturer", 
        "ElectrodeManufacturersModelName", 
    ],
    ignored_files=[],
)

print("The BIDS conversion has completed")
print(f"Your BIDS dataset contains {len(err)} errors and {len(warn)} warnings")